In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import Dataset, DataLoader
import torch

In [ ]:
batch_size = 16

In [ ]:
import json
import csv

with open('DescriptionsUN1-72.csv', mode='r', newline='', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    data = [row for row in reader]

In [ ]:
import requests
import time
from tqdm import tqdm

url = ""
MODEL = "Qwen/Qwen2.5-7B-Instruct"

In [ ]:
results = []

for sample in tqdm(data):
    user_input = sample['descr']

    payload = {
        "model": MODEL,
        "stream": False,
        "max_tokens": 3,
        "enable_thinking": False,
        "min_p": 0.05,
        "temperature": 0.7,
        "top_p": 0.7,
        "top_k": 50,
        "frequency_penalty": 0.5,
        "n": 1,
        "messages": [
                {"role": "system", 
                    "content": f"You are acting as the representative of a country in the United Nations General Assembly to vote YES, NO, or ABSTAIN on a resolution."},
                {"role": "user", 
                    "content": f"Given the following resolution, you must as a delegate decide whether your country votes:\n\n{user_input}\n\nOnly respond with one of: YES, NO, or ABSTAIN"}
            ]
    }
    headers = {
        "Authorization": "",
        "Content-Type": "application/json"
    }

    response = requests.request("POST", url, json=payload, headers=headers).json()
    counter = 0
    while 'choices' not in response:
        time.sleep(30)
        response = requests.request("POST", url, json=payload, headers=headers).json()
    pred = response['choices'][0]['message']['content']
    results.append({
                "rcid": sample["rcid"],
                "session": sample['session'],
                "response": pred
            })

In [ ]:
with open("un_vote_"+MODEL+".json", "w") as json_file:
  json.dump(results, json_file, indent=4)

# Compute Similarity

In [ ]:
import json

with open('un_vote.json','r') as file:
    ori_data = json.load(file)
    
with open('un_vote_qwen.json','r') as file:
    llm_data = json.load(file)
llm_data_by_rcid = {item["rcid"]: item for item in llm_data}

In [ ]:
vote_mapping = {
    "YES": 1,
    "ABSTAIN": 2,
    "NO": 3,
    # "ABSENT": 8,
    # "NOT A MEMBER": 9
}

In [ ]:
from sklearn.metrics import cohen_kappa_score

def compute_cohens_kappa(votes_a, votes_b):
    """
    Compute Cohen's Kappa between two sets of votes.
    
    Parameters:
        votes_a (list of int): First rater's votes (values in {1, 2, 3})
        votes_b (list of int): Second rater's votes (values in {1, 2, 3})
        
    Returns:
        float: Cohen's Kappa score
    """
    if len(votes_a) != len(votes_b):
        raise ValueError("Length of votes_a and votes_b must be equal.")
    
    # scikit-learn treats integers as labels, so we can pass them directly
    kappa = cohen_kappa_score(votes_a, votes_b, labels=[1, 2, 3])
    return kappa

In [ ]:
scores = {}
for country in ori_data.keys():
    ori_vote = []
    llm_vote = []
    
    for d in ori_data[country]:
        if int(d['vote']) <= 3:
            sample = llm_data_by_rcid.get(d['rcid'])
            if sample['response'] in vote_mapping.keys():
                ori_vote.append(int(d['vote']))
                llm_vote.append(vote_mapping[sample['response']])
            
    # score = s_score(ori_vote, llm_vote)
    score = compute_cohens_kappa(ori_vote, llm_vote)
    scores[country] = score

In [ ]:
sorted_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)
sorted_dict = dict(sorted_items)

with open("un_vote_kappa_gpt.json", "w") as json_file:
    json.dump(sorted_dict, json_file, indent=4)

# Plot

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.colors as mcolors

world = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_0_countries.zip"
)

In [ ]:
import json

with open('UN_iso_a3.json','r') as file:
    iso = json.load(file)
with open('un_vote_kappa_qwen.json','r') as file:
    ori_data = json.load(file)
    
data = {
    'iso_a3': [],
    'value': []
}
for cn in ori_data:
    if cn not in iso:
        continue
    data['iso_a3'].append(iso[cn])
    data['value'].append(ori_data[cn])
    
data = pd.DataFrame(data)

In [ ]:
data.iloc[[0]]

In [ ]:
merged = world.merge(data, left_on='ADM0_A3', right_on='iso_a3', how='left')

norm = mcolors.TwoSlopeNorm(vmin=-0.1, vcenter=0, vmax=0.35)
cmap = plt.cm.RdBu_r       # red = positive, blue = negative

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

merged.plot(
    column='value',
    cmap=cmap,
    linewidth=0.8,
    ax=ax,
    edgecolor='0.8',
    legend=True,
    norm=norm,
    missing_kwds={"color": "lightgrey", "hatch": "///", "label": "No data"}
)
centroids = merged.dropna(subset=['value']).copy()
top5 = centroids.nlargest(5, 'value')
bottom5 = centroids.nsmallest(5, 'value')
highlight = pd.concat([top5, bottom5])
highlight["color"] = highlight["value"].apply(lambda v: cmap(norm(v)))
highlight = highlight.copy()
highlight["geometry"] = highlight.geometry.centroid
highlight.plot(
    ax=ax,
    markersize=15,
    color=highlight["color"],
    alpha=0.9,
    edgecolor='black',
    linewidth=0.3
)

for _, row in highlight.iterrows():
    ax.text(
        row.geometry.x,
        row.geometry.y,
        row['iso_a3'],
        fontsize=7,
        ha='center',
        va='center',
        weight='bold',
        color='black',
        zorder=4
    )

ax.axis('off')
plt.show()

# Table

In [ ]:
import json

with open('UN_iso_a3.json','r') as file:
    iso = json.load(file)
with open('un_vote_kappa_qwen.json','r') as file:
    ori_data = json.load(file)
    
rank = []
for cn in ori_data:
    if cn not in iso:
        print(cn)
        continue
    rank.append(iso[cn])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

headers = ["1-25", "26-50", "51-75", "76-100", "101-125", "126-150", "151-175", "176-200"]

rows, ranks = [], []
for i in range(25):
    row, rank_row = [], []
    for j in range(8):
        index = j * 25 + i
        if index < len(rank):
            row.append(rank[index])
            rank_row.append(index + 1)
        else:
            row.append("")
            rank_row.append(np.nan)
    rows.append(row)
    ranks.append(rank_row)

ranks = np.array(ranks, dtype=float)
norm = (ranks - np.nanmin(ranks)) / (np.nanmax(ranks) - np.nanmin(ranks))

cmap = plt.cm.Oranges
colors = cmap(0.7 * (1 - norm)) 

fig, ax = plt.subplots(figsize=(8, 8))
ax.axis('off')

table = ax.table(
    cellText=rows,
    colLabels=headers,
    loc='center',
    cellLoc='center'
)

for i in range(25):
    for j in range(8):
        if not np.isnan(ranks[i, j]):
            table[(i + 1, j)].set_facecolor(colors[i, j])

for j in range(8):
    cell = table[(0, j)]
    cell.set_facecolor("#333232")
    cell.get_text().set_color("white")
    cell.get_text().set_weight("bold")

plt.tight_layout()
plt.savefig("ranked_table_lighter.png", dpi=300)
plt.show()
